# Generate Pipeline (Unified)

`vocab_alignment.ipynb`, `generate/recipe_molecule_pipeline_redesign.ipynb`, `analysis/cuisine_only_cluster_analysis.ipynb`의 핵심 로직을 하나로 정리한 노트북입니다.

## 목표
- 입력: `../recipes.csv`, `../preprocess/flavordb_alias_in_vocab_only.csv`, `../molecules.csv`
- 처리: FlavorDB2 재료를 **하나 이상 포함하는 레시피만** 대상으로 분석
- 출력:
  - cuisine별 + ALL 레벨 graph csv 저장
  - cuisine별 + ALL 분자 그래프 클러스터링
  - cluster lift 결과(csv) 및 시각화(plot)
  - cluster graph 시각화(plot)

## 핵심 수식
$$
S(r,m)=\sum_{i \in I(r)} w_{r,i}\cdot\frac{\mathbf{1}\{m \in M(i)\}}{|M(i)|}
$$

## 0) 환경 설정
- 긴 루프에는 `tqdm`을 사용합니다.
- 중간 산출물은 `shape`, `head(10)`을 출력합니다.

In [ ]:
from __future__ import annotations

import ast
import re
import time
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **kwargs):
        return x

plt.style.use('seaborn-v0_8-whitegrid')
START_TS = time.time()

In [ ]:
# Paths + export settings
RECIPES_PATH = Path('../recipes.csv')
FLAVORDB_PATH = Path('../preprocess/flavordb_alias_in_vocab_only.csv')
MOLECULES_PATH = Path('../molecules.csv')

OUT_DIR = Path('../result')
GRAPH_DIR = OUT_DIR / 'graph'
ANALYSIS_DIR = OUT_DIR / 'analysis'
PLOT_DIR = OUT_DIR / 'plots'

# refactoring point: later set to None / [] to export all cuisines
TARGET_CUISINES = ['Thai', 'Korean']
TOP_K_MOLECULES_PER_RECIPE = 200
EXPORT_ALL_GROUP = False

for d in [OUT_DIR, GRAPH_DIR, ANALYSIS_DIR, PLOT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Output directories:')
for d in [GRAPH_DIR, ANALYSIS_DIR, PLOT_DIR]:
    print(' -', d.resolve())
print('TARGET_CUISINES =', TARGET_CUISINES)
print('TOP_K_MOLECULES_PER_RECIPE =', TOP_K_MOLECULES_PER_RECIPE)
print('EXPORT_ALL_GROUP =', EXPORT_ALL_GROUP)


## 1) 데이터 로드 + 유틸 함수

In [ ]:
def norm_text(s: str) -> str:
    if pd.isna(s):
        return ''
    s = str(s).strip().lower()
    s = re.sub(r'[_\-]+', ' ', s)
    s = re.sub(r'\s+', ' ', s)
    s = re.sub(r"[^\w\s']", '', s)
    return s.strip()


def find_col(df: pd.DataFrame, candidates: list[str]) -> str | None:
    lower_map = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    return None


def parse_ingredient_ratio(raw):
    if pd.isna(raw):
        return []

    if isinstance(raw, str):
        txt = raw.strip()
        if not txt:
            return []
        try:
            raw = ast.literal_eval(txt)
        except Exception:
            return []

    out = []
    if isinstance(raw, dict):
        for k, v in raw.items():
            try:
                out.append((norm_text(k), float(v)))
            except Exception:
                continue
    elif isinstance(raw, list):
        for item in raw:
            if isinstance(item, dict):
                ing = item.get('ingredient', item.get('name', item.get('ing')))
                val = item.get('ratio', item.get('weight', item.get('w', item.get('gram', 0))))
                try:
                    out.append((norm_text(ing), float(val)))
                except Exception:
                    continue
            elif isinstance(item, (list, tuple)) and len(item) >= 2:
                try:
                    out.append((norm_text(item[0]), float(item[1])))
                except Exception:
                    continue

    return [(i, w) for i, w in out if i and np.isfinite(w) and w > 0]

In [ ]:
recipes = pd.read_csv(RECIPES_PATH)
flavordb = pd.read_csv(FLAVORDB_PATH)
molecules = pd.read_csv(MOLECULES_PATH)

print('recipes:', recipes.shape)
print('flavordb:', flavordb.shape)
print('molecules:', molecules.shape)

display(recipes.head(3))
display(flavordb.head(3))
display(molecules.head(3))

In [ ]:
recipe_id_col = find_col(recipes, ['recipe_id', 'id', 'rid'])
recipe_name_col = find_col(recipes, ['name', 'recipe_name', 'title'])
cuisine_col   = find_col(recipes, ['cuisine', 'cuisine_name'])
ratio_col     = find_col(recipes, ['ingredients_ratio', 'ingredient_ratio', 'ingredients_with_ratio'])
cleaned_col   = find_col(recipes, ['cleaned_ingredients', 'ingredients'])

if cuisine_col is None:
    raise KeyError(f"Missing cuisine column. columns={list(recipes.columns)}")

if ratio_col is None and cleaned_col is None:
    raise KeyError(f"Missing both ratio and cleaned ingredient columns. columns={list(recipes.columns)}")

if recipe_id_col is None:
    recipes = recipes.copy()
    recipes['recipe_id'] = np.arange(recipes.shape[0], dtype=np.int64)
    recipe_id_col = 'recipe_id'

if recipe_name_col is None:
    recipes = recipes.copy()
    recipes['name'] = recipes[recipe_id_col].map(lambda x: f'recipe_{x}')
    recipe_name_col = 'name'

ing_col = find_col(flavordb, ['alias', 'ingredient', 'name', 'entity_alias_readable', 'entity_alias'])
mol_col = find_col(flavordb, ['molecules', 'molecule_ids', 'molecule id', 'molecule_id'])

if ing_col is None or mol_col is None:
    raise KeyError(f'FlavorDB columns missing. ing_col={ing_col}, mol_col={mol_col}, all={list(flavordb.columns)}')

molecule_id_col = find_col(molecules, ['pubchem id', 'molecule', 'molecule_id', 'id'])
molecule_name_col = find_col(molecules, ['common name', 'molecule_name', 'name'])
flavor_profile_col = find_col(molecules, ['flavor profile', 'flavor_profile'])

if molecule_id_col is None:
    raise KeyError(f'Molecule id column missing. all={list(molecules.columns)}')

print('recipe_id_col:', recipe_id_col)
print('recipe_name_col:', recipe_name_col)
print('cuisine_col:', cuisine_col)
print('ratio_col:', ratio_col)
print('flavor ingredient col:', ing_col)
print('flavor molecule col:', mol_col)
print('molecule_id_col:', molecule_id_col)
print('molecule_name_col:', molecule_name_col)
print('flavor_profile_col:', flavor_profile_col)


## 2) FlavorDB ingredient → molecule set $M(i)$ 생성

In [ ]:
import ast
import re

def parse_molecule_ids(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return []
    if isinstance(x, (set, list, tuple)):
        return [int(v) for v in x]
    s = str(x).strip()
    if s == "" or s.lower() in {"nan", "none"}:
        return []
    # "{1, 2, 3}" / "[1,2,3]" 같은 경우
    try:
        val = ast.literal_eval(s)
        if isinstance(val, (set, list, tuple)):
            return [int(v) for v in val]
    except Exception:
        pass
    # fallback: 문자열에서 숫자만 뽑기
    return [int(v) for v in re.findall(r"\d+", s)]

In [ ]:
flavordb = flavordb.copy()
flavordb['ingredient_norm'] = flavordb[ing_col].map(norm_text)
flavordb['molecule_ids'] = flavordb[mol_col].map(parse_molecule_ids)

tmp = flavordb[['ingredient_norm', 'molecule_ids']].dropna(subset=['ingredient_norm'])
tmp = tmp.explode('molecule_ids').dropna(subset=['molecule_ids'])

ing_to_molecules = (
    tmp.groupby('ingredient_norm')['molecule_ids']
      .apply(lambda x: sorted(set(int(v) for v in x)))
      .to_dict()
)

print('mapped ingredients:', len(ing_to_molecules))
print('sample mapping:', list(ing_to_molecules.items())[:5])

## 3) 레시피 long format 생성 + FlavorDB2 포함 레시피 필터링

In [ ]:
if TARGET_CUISINES:
    recipes = recipes[recipes[cuisine_col].isin(TARGET_CUISINES)].copy()
    print('recipes filtered by cuisine:', recipes.shape)

records = []
for _, row in tqdm(recipes.iterrows(), total=len(recipes), desc='Parse recipes'):
    rid = row[recipe_id_col]
    rname = row[recipe_name_col]
    cuisine = row[cuisine_col]
    pairs = parse_ingredient_ratio(row[ratio_col])

    z = sum(w for _, w in pairs)
    if z <= 0:
        continue

    for ing, w in pairs:
        records.append({
            'recipe_id': int(rid),
            'name': str(rname),
            'cuisine': str(cuisine),
            'ingredient': ing,
            'w_ri': float(w / z),
        })

recipes_long = pd.DataFrame(records)
print('recipes_long shape:', recipes_long.shape)
display(recipes_long.head(10))

if recipes_long.empty:
    raise RuntimeError('No parsed recipe rows. Check ratio_col parsing logic and input file format.')

recipes_long['has_flavordb_match'] = recipes_long['ingredient'].isin(ing_to_molecules)
matched_rids = set(recipes_long.loc[recipes_long['has_flavordb_match'], 'recipe_id'].unique())
recipes_long = recipes_long[recipes_long['recipe_id'].isin(matched_rids)].copy()

print('filtered recipes_long shape:', recipes_long.shape)
print('n_recipes_with_flavordb_ingredient:', recipes_long['recipe_id'].nunique())
display(recipes_long.head(10))

recipes_long.to_csv(OUT_DIR / 'recipes_long_normalized.csv', index=False)


## 4) 수식 기반 recipe-molecule score 계산

아래 수식으로 $S(r,m)$을 계산합니다.
$$
S(r,m)=\sum_{i \in I(r)} w_{r,i}\cdot\frac{\mathbf{1}\{m \in M(i)\}}{|M(i)|}
$$

In [ ]:
score_rows = []
unk_rows = []

for rid, grp in tqdm(recipes_long.groupby('recipe_id'), desc='Compute S(r,m)'):
    acc = defaultdict(float)
    mass_covered = 0.0

    for ing, w in grp[['ingredient', 'w_ri']].itertuples(index=False):
        mols = ing_to_molecules.get(ing, [])
        if not mols:
            continue
        contrib = w / len(mols)
        for m in mols:
            acc[m] += contrib
        mass_covered += w

    for m, s in acc.items():
        score_rows.append({'recipe_id': int(rid), 'molecule': int(m), 'S_rm': float(s)})

    unk_rows.append({'recipe_id': int(rid), 'u_r': float(max(0.0, 1.0 - mass_covered))})

recipe_molecule = pd.DataFrame(score_rows)
recipe_unk = pd.DataFrame(unk_rows)

rid_meta = (
    recipes_long[['recipe_id', 'name', 'cuisine']]
    .drop_duplicates(subset=['recipe_id'])
)

mol_meta_cols = [molecule_id_col]
if molecule_name_col is not None:
    mol_meta_cols.append(molecule_name_col)
if flavor_profile_col is not None:
    mol_meta_cols.append(flavor_profile_col)

molecule_meta = molecules[mol_meta_cols].copy()
rename_map = {molecule_id_col: 'molecule'}
if molecule_name_col is not None:
    rename_map[molecule_name_col] = 'molecule_name'
if flavor_profile_col is not None:
    rename_map[flavor_profile_col] = 'flavor_profile'
molecule_meta = molecule_meta.rename(columns=rename_map)
molecule_meta['molecule'] = pd.to_numeric(molecule_meta['molecule'], errors='coerce')
molecule_meta = molecule_meta.dropna(subset=['molecule']).copy()
molecule_meta['molecule'] = molecule_meta['molecule'].astype(int)
molecule_meta = molecule_meta.drop_duplicates(subset=['molecule'])

recipe_molecule = recipe_molecule.merge(rid_meta, on='recipe_id', how='left')
recipe_molecule = recipe_molecule.merge(molecule_meta, on='molecule', how='left')

recipe_molecule = recipe_molecule.sort_values(['recipe_id', 'S_rm'], ascending=[True, False]).copy()
recipe_molecule['rank'] = recipe_molecule.groupby('recipe_id')['S_rm'].rank(method='first', ascending=False).astype(int)
recipe_molecule = recipe_molecule[recipe_molecule['rank'] <= TOP_K_MOLECULES_PER_RECIPE].copy()
recipe_molecule['p'] = recipe_molecule['S_rm'] / recipe_molecule.groupby('recipe_id')['S_rm'].transform('sum')

print('recipe_molecule (top-k + normalized):', recipe_molecule.shape)
print('recipe_unk:', recipe_unk.shape)
display(recipe_molecule.head(10))
display(recipe_unk.head(10))

recipe_molecule.to_csv(GRAPH_DIR / 'recipe_molecule_edges.csv', index=False)
recipe_unk.to_csv(GRAPH_DIR / 'recipe_unk_mass.csv', index=False)


## 5) Cuisine별 + ALL graph csv 저장

In [ ]:
def safe_key(name: str) -> str:
    return re.sub(r'\W+', '_', str(name)).strip('_') or 'unknown'


def export_graph_for_group(df_edges: pd.DataFrame, key: str):
    cuisine_dir = GRAPH_DIR / key
    cuisine_dir.mkdir(parents=True, exist_ok=True)

    edge_cols = ['recipe_id', 'name', 'rank', 'molecule', 'molecule_name', 'flavor_profile', 'p']
    edge_cols = [c for c in edge_cols if c in df_edges.columns]
    edge_export = df_edges[edge_cols].sort_values(['recipe_id', 'rank']).copy()
    edge_export.to_csv(cuisine_dir / '000_recipe_molecule_edges.csv', index=False)

    mol_weight = (
        df_edges.groupby('molecule', as_index=False)['p']
        .sum()
        .rename(columns={'p': 'weight'})
    )
    mol_meta = df_edges[['molecule', 'molecule_name', 'flavor_profile']].drop_duplicates(subset=['molecule'])
    mol_weight = mol_weight.merge(mol_meta, on='molecule', how='left')
    mol_weight = mol_weight.sort_values('weight', ascending=False).copy()
    mol_weight['rank'] = np.arange(1, len(mol_weight) + 1)
    mol_weight.to_csv(cuisine_dir / '001_molecule_weight.csv', index=False)

    mol_recipe = df_edges[['molecule', 'molecule_name', 'flavor_profile', 'recipe_id', 'name', 'p']].copy()
    mol_recipe = mol_recipe.sort_values(['molecule', 'p'], ascending=[True, False]).copy()
    mol_recipe['rank'] = mol_recipe.groupby('molecule')['p'].rank(method='first', ascending=False).astype(int)
    mol_recipe = mol_recipe[['molecule', 'molecule_name', 'flavor_profile', 'rank', 'recipe_id', 'name', 'p']]
    mol_recipe.to_csv(cuisine_dir / '002_molecule_recipe_edges.csv', index=False)

    print(f'[{key}] edges={edge_export.shape}, molecule_weight={mol_weight.shape}, molecule_recipe={mol_recipe.shape}')
    display(edge_export.head(10))
    display(mol_weight.head(10))
    display(mol_recipe.head(10))


for cuisine, sub in tqdm(recipe_molecule.groupby('cuisine'), desc='Export cuisine graphs'):
    export_graph_for_group(sub.copy(), safe_key(cuisine))

if EXPORT_ALL_GROUP:
    export_graph_for_group(recipe_molecule.copy(), 'ALL')


## 6) Molecule graph 구성 + clustering + ingredient lift + 시각화

In [ ]:
def build_recipe_to_molset(df_edges: pd.DataFrame):
    d = defaultdict(set)
    for rid, mid in df_edges[['recipe_id', 'molecule']].itertuples(index=False):
        d[int(rid)].add(int(mid))
    return d


def build_molecule_graph(rids, rid2molset, min_cooc=3):
    cooc = Counter()
    for rid in rids:
        mols = sorted(rid2molset.get(int(rid), []))
        for i in range(len(mols)):
            for j in range(i + 1, len(mols)):
                a, b = mols[i], mols[j]
                cooc[(a, b)] += 1

    G = nx.Graph()
    for (a, b), w in cooc.items():
        if w >= min_cooc:
            G.add_edge(a, b, weight=w)
    return G


def cluster_graph(G: nx.Graph):
    if G.number_of_nodes() == 0:
        return {}
    communities = nx.algorithms.community.louvain_communities(G, weight='weight', seed=42)
    part = {}
    for cid, comm in enumerate(communities):
        for node in comm:
            part[node] = cid
    return part


def ingredient_lift(recipes_subset: pd.DataFrame, recipe_ids_set: set, top_n=15):
    grp = recipes_subset.groupby('ingredient')['recipe_id'].nunique().rename('in_cluster')
    bg = recipes_subset.groupby('ingredient')['recipe_id'].nunique().rename('in_all')
    df = pd.concat([grp, bg], axis=1).fillna(0)
    df['p_cluster'] = df['in_cluster'] / max(len(recipe_ids_set), 1)
    df['p_all'] = df['in_all'] / max(recipes_subset['recipe_id'].nunique(), 1)
    df['lift'] = df['p_cluster'] / df['p_all'].replace(0, np.nan)
    df = df.sort_values(['lift', 'in_cluster'], ascending=False).head(top_n)
    return df.reset_index()


In [ ]:
rid2molset = build_recipe_to_molset(recipe_molecule)
all_cuisines = sorted(recipes_long['cuisine'].dropna().unique().tolist())
target_groups = all_cuisines.copy()
if EXPORT_ALL_GROUP:
    target_groups = ['ALL'] + target_groups

analysis_summary = []

for group in tqdm(target_groups, desc='Cuisine clustering'):
    if group == 'ALL':
        rids = sorted(recipes_long['recipe_id'].unique())
    else:
        rids = sorted(recipes_long.loc[recipes_long['cuisine'] == group, 'recipe_id'].unique())

    print(f'==== {group} ====')
    print('recipes:', len(rids))

    G = build_molecule_graph(rids, rid2molset, min_cooc=3)
    part = cluster_graph(G)

    n_clusters = len(set(part.values())) if part else 0
    print('graph nodes:', G.number_of_nodes(), 'edges:', G.number_of_edges(), 'clusters:', n_clusters)

    fig = plt.figure(figsize=(8, 6))
    if G.number_of_nodes() > 0:
        pos = nx.spring_layout(G, seed=42)
        node_colors = [part.get(n, -1) for n in G.nodes()]
        nx.draw_networkx(
            G,
            pos=pos,
            node_size=80,
            with_labels=False,
            edge_color='lightgray',
            node_color=node_colors,
            cmap=plt.cm.tab20,
        )
    plt.title(f'Molecule graph clusters - {group}')
    plt.axis('off')
    fig.savefig(PLOT_DIR / f'clusters_{safe_key(group)}.png', dpi=200, bbox_inches='tight')
    plt.close(fig)

    analysis_summary.append({
        'group': group,
        'n_recipes': len(rids),
        'n_nodes': G.number_of_nodes(),
        'n_edges': G.number_of_edges(),
        'n_clusters': n_clusters,
    })


## 7) 요약 저장

In [ ]:
summary_df = pd.DataFrame(analysis_summary)
summary_df.to_csv(ANALYSIS_DIR / 'analysis_summary.csv', index=False)

print('analysis summary:', summary_df.shape)
display(summary_df.head(10))

elapsed = time.time() - START_TS
print(f'Done. elapsed={elapsed:.1f}s')
print('Check outputs in:')
print(' -', GRAPH_DIR.resolve())
print(' -', ANALYSIS_DIR.resolve())
print(' -', PLOT_DIR.resolve())